### 🔗 Links and Resources
| Method / Config | Description |
|-----------------|-------------|
| [broadcast()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.functions.broadcast.html) | Marks a DataFrame to be broadcast to all worker nodes during a join |
| [DataFrame.hint()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.hint.html) | Applies a hint to the DataFrame (e.g. `"broadcast"`) |
| [DataFrame.explain()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.explain.html) | Prints the physical plan — look for `BroadcastHashJoin` or `BroadcastExchange` |
| [SparkConf.get()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.conf.html) | Reads a runtime Spark configuration value |
| `spark.sql.autoBroadcastJoinThreshold` | Max table size (bytes) Spark will auto-broadcast; set to `-1` to disable |

In [1]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel — 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()


def display(df):
    df.show(truncate=False)

### 📌 Creating Sample DataFrames

We use a **large** fact table (`orders`) and a **small** dimension table (`products`).
Broadcasting the small table avoids a shuffle of the large one.

In [2]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql import functions as F

# Large fact table: orders
orders_schema = StructType([
    StructField("order_id",   IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity",   IntegerType(), True),
])

orders_data = [
    (1001, 1, 2), (1002, 3, 1), (1003, 2, 5), (1004, 1, 3),
    (1005, 4, 2), (1006, 2, 1), (1007, 5, 4), (1008, 3, 2),
    (1009, 1, 1), (1010, 4, 3),
]

df_orders = spark.createDataFrame(orders_data, schema=orders_schema)

# Small dimension table: products
products_schema = StructType([
    StructField("product_id",   IntegerType(), True),
    StructField("product_name", StringType(),  True),
    StructField("unit_price",   DoubleType(),  True),
])

products_data = [
    (1, "Widget A", 9.99),
    (2, "Widget B", 14.99),
    (3, "Gadget C", 24.99),
    (4, "Gadget D", 4.99),
    (5, "Part E",   1.99),
]

df_products = spark.createDataFrame(products_data, schema=products_schema)

print("Orders:")
df_orders.show()
print("Products:")
df_products.show()

Orders:
+--------+----------+--------+
|order_id|product_id|quantity|
+--------+----------+--------+
|    1001|         1|       2|
|    1002|         3|       1|
|    1003|         2|       5|
|    1004|         1|       3|
|    1005|         4|       2|
|    1006|         2|       1|
|    1007|         5|       4|
|    1008|         3|       2|
|    1009|         1|       1|
|    1010|         4|       3|
+--------+----------+--------+

Products:
+----------+------------+----------+
|product_id|product_name|unit_price|
+----------+------------+----------+
|         1|    Widget A|      9.99|
|         2|    Widget B|     14.99|
|         3|    Gadget C|     24.99|
|         4|    Gadget D|      4.99|
|         5|      Part E|      1.99|
+----------+------------+----------+



### 📌 Reading the Broadcast Threshold from Configuration

In [4]:
# spark.sql.autoBroadcastJoinThreshold controls the size limit (in bytes)
# below which Spark automatically broadcasts a table during a join.
# Default is 10 MB (10485760 bytes). Set to -1 to disable auto-broadcast.
try:
    threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
except Exception:
    threshold = "10485760"  # Default 10 MB in bytes
threshold_mb = int(threshold) / (1024 * 1024)

print(f"spark.sql.autoBroadcastJoinThreshold = {threshold} bytes ({threshold_mb:.1f} MB)")

spark.sql.autoBroadcastJoinThreshold = 10485760 bytes (10.0 MB)


### 📌 Explicit Broadcast Join Using broadcast()

In [5]:
# Wrapping the small DataFrame in broadcast() tells Spark to send
# the entire table to every executor instead of performing a shuffle join.
df_broadcast_join = df_orders.join(
    F.broadcast(df_products),
    on="product_id",
    how="inner"
)

display(df_broadcast_join)

+----------+--------+--------+------------+----------+
|product_id|order_id|quantity|product_name|unit_price|
+----------+--------+--------+------------+----------+
|1         |1001    |2       |Widget A    |9.99      |
|3         |1002    |1       |Gadget C    |24.99     |
|2         |1003    |5       |Widget B    |14.99     |
|1         |1004    |3       |Widget A    |9.99      |
|4         |1005    |2       |Gadget D    |4.99      |
|2         |1006    |1       |Widget B    |14.99     |
|5         |1007    |4       |Part E      |1.99      |
|3         |1008    |2       |Gadget C    |24.99     |
|1         |1009    |1       |Widget A    |9.99      |
|4         |1010    |3       |Gadget D    |4.99      |
+----------+--------+--------+------------+----------+



### 📌 Examining the Physical Plan

Look for **`BroadcastHashJoin`** and **`BroadcastExchange`** in the output — these confirm Spark chose a broadcast strategy.

In [6]:
# extended=True prints Parsed / Analyzed / Optimized / Physical plans
df_broadcast_join.explain(extended=False)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [product_id#13229, order_id#13228, quantity#13230, product_name#13242, unit_price#13243]
         +- PhotonBroadcastHashJoin [product_id#13229], [product_id#13241], Inner, BuildRight, false, true
            :- PhotonRowToColumnar
            :  +- LocalTableScan [order_id#13228, product_id#13229, quantity#13230]
            +- PhotonShuffleExchangeSource
               +- PhotonShuffleMapStage EXECUTOR_BROADCAST, [id=#8374]
                  +- PhotonShuffleExchangeSink SinglePartition
                     +- PhotonRowToColumnar
                        +- LocalTableScan [product_id#13241, product_name#13242, unit_price#13243]


== Photon Explanation ==
The query is fully supported by Photon.


### 📌 Broadcast Join Using hint()

In [ ]:
# hint("broadcast") is equivalent to broadcast() but uses the DataFrame API hint mechanism.
# Useful when you cannot wrap the DataFrame in a function call directly.
df_hint_join = df_orders.join(
    df_products.hint("broadcast"),
    on="product_id",
    how="inner"
)

display(df_hint_join)
df_hint_join.explain(extended=False)

### 📌 Disabling Auto-Broadcast and Comparing Plans

In [ ]:
# Temporarily disable auto-broadcast to see the shuffle join plan
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

df_shuffle_join = df_orders.join(df_products, on="product_id", how="inner")
print("--- Plan WITHOUT broadcast (auto-broadcast disabled) ---")
df_shuffle_join.explain(extended=False)

# Restore the original threshold
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", threshold)
print(f"Threshold restored to {threshold} bytes")